# Two stage classification predictor

The notebook detils the Two-stage classification pipeline, including hyperparameter tuning, test set predictions, evaluation of model performance, and permutation importance.
The resulting best predictive models following hyperparameter tuning, are initialized in section 2b. with the respective optimal hyperparamters. 

## Table of Contents

- [1. Baseline Estimators ](#1.-Baseline-estimators)
- [2. Tuning](#2.-Tuning)
- [2b. Initializing best performing models](#3.-Initializing-the-best-perfoming-models-with-the-optimized-hyperparameters)
- [3. Evaluation on Test Set](#3.-Test)
- [4. Permutation Importance](#3.-Permutation-importance)

In [ ]:
import os
import sys
from pathlib import Path

# Find repository root
ROOT = Path(os.getcwd())

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Common paths
DATA = ROOT / "data"
SCRIPTS = ROOT / "scripts"

sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_curve, precision_recall_curve, auc, make_scorer, recall_score, roc_auc_score, matthews_corrcoef, classification_report
from sklearn.inspection import permutation_importance

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV,learning_curve, train_test_split, cross_validate

In [ ]:
import scripts.ML.ML_scripts as ml

### Load Data

In [ ]:
train_data = pd.read_pickle(DATA/'ml_data/L2_TRAIN.pkl')

features = train_data.iloc[:, :10]


target = train_data[['source']].copy()
train_data['source'] = train_data['source'].str.split('_').str[0] #less specific nitrogen source
train_data['source_type'] = train_data['source'].str.contains(r'arg|nh4|no3', case=False, na=False).map({True: 'nitrogen', False: 'carbon'})

**LABEL ENCODING**

In [ ]:
# ---- fit encoders ----
le_type     = LabelEncoder().fit(train_data['source_type'])
le_source   = LabelEncoder().fit(train_data['source'])
le_carbon   = LabelEncoder().fit(train_data.loc[train_data['source_type'] == 'carbon',   'source'])
le_nitrogen = LabelEncoder().fit(train_data.loc[train_data['source_type'] == 'nitrogen', 'source'])

# ---- encode labels ----
y_type   = le_type.transform(train_data['source_type'])
y_source = le_source.transform(train_data['source'])

# ---- masks ----
carbon_mask   = train_data['source_type'] == 'carbon'
nitrogen_mask = train_data['source_type'] == 'nitrogen'

# ---- features ----
X = features

# ---- stage 2 subsets ----
X_c = X[carbon_mask.values]
X_n = X[nitrogen_mask.values]

y_c = le_carbon.transform(train_data.loc[carbon_mask, 'source'])
y_n = le_nitrogen.transform(train_data.loc[nitrogen_mask, 'source'])

## 1. Baseline estiamtors

**RandomForest**

In [ ]:
rf_binary = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

rf_carbon = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

rf_nitrogen = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

**XGBoost**

In [ ]:
xgb_binary = XGBClassifier(
    random_state=42,
    n_jobs=1,
    verbosity=0
)

xgb_carbon = XGBClassifier(
    random_state=42,
    n_jobs=1,
    verbosity=0
)

xgb_nitrogen = XGBClassifier(
    random_state=42,
    n_jobs=1,
    verbosity=0
)

## 2. Tuning

Parameter grid:

In [ ]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth':[5, 10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

cb_params = {
    'iterations': [100, 300, 500],
    'learning_rate': [0.01, 0.03, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [3, 5, 7]
}

xgb_params = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.03, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.5, 0.8],
    'reg_lambda': [1, 3, 5]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "mcc": make_scorer(matthews_corrcoef)
}

In [ ]:
def tune_model(model, param_grid, X, y, model_name, stage_name, refit_score="f1_macro"):
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        refit=refit_score,  
        cv=cv,
        n_jobs=3,
        return_train_score=False
    )

    search.fit(X, y)

    i = search.best_index_

    result = {
        "stage": stage_name,
        "model": model_name,
        "best_params": search.best_params_,
        "accuracy": search.cv_results_["mean_test_accuracy"][i],
        "precision_macro": search.cv_results_["mean_test_precision_macro"][i],
        "recall_macro": search.cv_results_["mean_test_recall_macro"][i],
        "f1_macro": search.cv_results_["mean_test_f1_macro"][i],
        "mcc": search.cv_results_["mean_test_mcc"][i],
        "best_estimator": search.best_estimator_
    }

    return result

### Run tuner:

In [ ]:
results = []

# --------------------
# Stage 1: source_type
# --------------------

results.append(
    tune_model(
        rf_binary,
        rf_params,
        X,
        y_type,
        "Random Forest",
        "Stage 1: source_type"
    )
)

results.append(
    tune_model(
        xgb_binary,
        xgb_params,
        X,
        y_type,
        "XGBoost",
        "Stage 1: source_type",
        refit_score="mcc"
    )
)


# --------------------
# Stage 2: carbon source
# --------------------

results.append(
    tune_model(
        rf_carbon,
        rf_params,
        X_c,
        y_c,
        "Random Forest",
        "Stage 2: carbon source"
    )
)

results.append(
    tune_model(
        xgb_carbon,
        xgb_params,
        X_c,
        y_c,
        "XGBoost",
        "Stage 2: carbon source"
    )
)


# --------------------
# Stage 2: nitrogen source
# --------------------

results.append(
    tune_model(
        rf_nitrogen,
        rf_params,
        X_n,
        y_n,
        "Random Forest",
        "Stage 2: nitrogen source"
    )
)

results.append(
    tune_model(
        xgb_nitrogen,
        xgb_params,
        X_n,
        y_n,
        "XGBoost",
        "Stage 2: nitrogen source"
    )
)

In [ ]:
for r in results:
    print("\n", r["stage"], "-", r["model"])
    print(r["best_params"])

In [ ]:
results_df = pd.DataFrame(results)
metrics_df = results_df.drop(columns=["best_estimator", "best_params"])
metrics_df

In [ ]:
best_rf_binary = results_df.query(
    "stage == 'Stage 1: source_type' and model == 'Random Forest'"
)["best_estimator"].iloc[0]

best_xgb_binary = results_df.query(
    "stage == 'Stage 1: source_type' and model == 'XGBoost'"
)["best_estimator"].iloc[0]

best_rf_carbon = results_df.query(
    "stage == 'Stage 2: carbon source' and model == 'Random Forest'"
)["best_estimator"].iloc[0]

best_xgb_carbon = results_df.query(
    "stage == 'Stage 2: carbon source' and model == 'XGBoost'"
)["best_estimator"].iloc[0]

best_rf_nitrogen = results_df.query(
    "stage == 'Stage 2: nitrogen source' and model == 'Random Forest'"
)["best_estimator"].iloc[0]

best_xgb_nitrogen = results_df.query(
    "stage == 'Stage 2: nitrogen source' and model == 'XGBoost'"
)["best_estimator"].iloc[0]

In [ ]:
for r in results:
    print("\n", r["stage"], "-", r["model"])
    print(r["best_params"])

 ```text
 Stage 1: source_type - XGBoost
{'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 500, 'reg_lambda': 1, 'subsample': 0.8}

 Stage 2: carbon source - XGBoost
{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'reg_lambda': 5, 'subsample': 0.5}

 Stage 2: nitrogen source - XGBoost
{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'reg_lambda': 5, 'subsample': 0.8}

## 2b. Initializing the best perfoming models with the optimized hyperparameters

In [ ]:
best_rf_binary = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    max_depth=20,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=200
)

best_rf_carbon = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=300
)

best_rf_nitrogen = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    max_depth=None,
    min_samples_leaf=2,
    min_samples_split=5,
    n_estimators=100
)

In [ ]:
best_xgb_binary = XGBClassifier(
    random_state=42,
    learning_rate=0.03,
    max_depth=7,
    n_estimators=500,
    reg_lambda=1,
    subsample=0.8
)

best_xgb_carbon = XGBClassifier(
    random_state=42,
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    reg_lambda=5,
    subsample=0.5
)

best_xgb_nitrogen = XGBClassifier(
    random_state=42,
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    reg_lambda=5,
    subsample=0.8
)

In [ ]:
best_rf_binary.fit(X, y_type)
best_rf_carbon.fit(X_c, y_c)
best_rf_nitrogen.fit(X_n, y_n)

best_xgb_binary.fit(X, y_type)
best_xgb_carbon.fit(X_c, y_c)
best_xgb_nitrogen.fit(X_n, y_n)

# 3. Test

Load test data:

In [ ]:
test_data = pd.read_pickle(DATA/'ml_data/L2_IID_TEST.pkl')

features_test = test_data.iloc[:, :10]

test_data['source'] = test_data['source'].str.split('_').str[0] #less specific nitrogen source
test_data['source_type'] = test_data['source'].str.contains(r'arg|nh4|no3', case=False, na=False).map({True: 'nitrogen', False: 'carbon'})

In [ ]:
y_test_type = le_type.transform(test_data["source_type"])
y_test_source = test_data["source"]

X_test = features_test

Label encoders:

In [ ]:
carbon_mask_test = test_data["source_type"] == "carbon"
nitrogen_mask_test = test_data["source_type"] == "nitrogen"

X_c_test = X_test[carbon_mask_test.values]
X_n_test = X_test[nitrogen_mask_test.values]

y_c_test = le_carbon.transform(
    test_data.loc[carbon_mask_test, "source"]
)

y_n_test = le_nitrogen.transform(
    test_data.loc[nitrogen_mask_test, "source"]
)

**MAKE PREDICTION**

In [ ]:
rf_binary_pred, rf_source_pred = ml.make_prediction(
    X_test,
    best_rf_binary,
    best_rf_carbon,
    best_rf_nitrogen,
    le_type,
    le_carbon,
    le_nitrogen
)

xgb_binary_pred, xgb_source_pred = ml.make_prediction(
    X_test,
    best_xgb_binary,
    best_xgb_carbon,
    best_xgb_nitrogen,
    le_type,
    le_carbon,
    le_nitrogen
)

**Stage 1**

In [ ]:
ml.evaluate_predictions(
    y_test_type,
    rf_binary_pred,
    "RF pipeline: Stage 1 source_type"
)
ml.evaluate_predictions(
    y_test_type,
    xgb_binary_pred,
    "XGBoost pipeline: Stage 1 source_type"
)

**NITROGEN = TRUE**

In [ ]:
rf_binary_scores = best_rf_binary.predict_proba(X_test)[:, 1]
xgb_binary_scores = best_xgb_binary.predict_proba(X_test)[:, 1]

In [ ]:
ml.plot_binary_eval_curves(
    y_test_type,
    xgb_binary_scores,
    model='XGBoost',
    figsize=(18,5)
)

In [ ]:
ml.plot_binary_eval_curves(
    y_test_type,
    rf_binary_scores,
    model='Random Forest'
)

In [ ]:
cm = confusion_matrix(y_test_type, rf_binary_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le_type.classes_)

# disp.plot(cmap='GnBu')
# plt.title("Stage 1: Random Forest")
# plt.show()

fig, ax = plt.subplots(figsize=(10,8))
disp.plot(cmap='GnBu', ax=ax)
# ax.set_title("Stage 1: Random Forest", fontsize=15, pad=15)
ax.text(-0.18, 1.05, "A", transform=ax.transAxes, fontsize=25, fontweight="bold", va="top")
ax.set_xlabel("Predicted label", fontsize=13, labelpad=15)
ax.set_ylabel("True label", fontsize=13, labelpad=15)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=12)
plt.show()

In [ ]:
cm = confusion_matrix(y_test_type, xgb_binary_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le_type.classes_)

fig, ax = plt.subplots(figsize=(10,8))
disp.plot(cmap='GnBu', ax=ax)
# ax.set_title("Stage 1: XGBoost", fontsize=15, pad=15)
ax.text(-0.18, 1.05, "B", transform=ax.transAxes, fontsize=25, fontweight="bold", va="top")
ax.set_xlabel("Predicted label", fontsize=13, labelpad=15)
ax.set_ylabel("True label", fontsize=13, labelpad=15)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=12)
plt.show()

**Stage 2**

In [ ]:
def evaluate_stage2_predictions(y_true, y_pred, name, label_encoder):
    print(f"\n=== {name} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print(
        "Precision:",
        precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    print(
        "Recall   :",
        recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    print(
        "F1       :",
        f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    print(
        "MCC      :",
        matthews_corrcoef(y_true, y_pred)
    )

    print("\nClassification report:")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )

In [ ]:
rf_c_pred = best_rf_carbon.predict(X_c_test)
rf_n_pred = best_rf_nitrogen.predict(X_n_test)

xgb_c_pred = best_xgb_carbon.predict(X_c_test)
xgb_n_pred = best_xgb_nitrogen.predict(X_n_test)

In [ ]:
ml.evaluate_stage2_predictions(
    y_c_test,
    rf_c_pred,
    "RF Stage 2 carbon",
    le_carbon
)

ml.evaluate_stage2_predictions(
    y_n_test,
    rf_n_pred,
    "RF Stage 2 nitrogen",
    le_nitrogen
)

In [ ]:
ml.evaluate_stage2_predictions(
    y_c_test,
    xgb_c_pred,
    "XGBBoost Stage 2 carbon",
    le_carbon
)

ml.evaluate_stage2_predictions(
    y_n_test,
    xgb_n_pred,
    "XGBBoost Stage 2 nitrogen",
    le_nitrogen
)

In [ ]:
c_order = ['glc', 'ac', 'lcts', 'glyc', 'succ']
n_order = ['arg', 'nh4', 'no3']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cm_carbon = confusion_matrix(le_carbon.inverse_transform(y_c_test), le_carbon.inverse_transform(rf_c_pred), labels=c_order)
disp_carbon = ConfusionMatrixDisplay(cm_carbon, display_labels=c_order)

disp_carbon.plot(
    cmap='GnBu',
    ax=axes[0],
    colorbar=True
)

axes[0].set_title("Carbon")

cm_nitrogen = confusion_matrix(le_nitrogen.inverse_transform(y_n_test), le_nitrogen.inverse_transform(rf_n_pred), labels=n_order)
disp_nitrogen = ConfusionMatrixDisplay(cm_nitrogen,display_labels=n_order)

disp_nitrogen.plot(cmap='GnBu',ax=axes[1],colorbar=True)

axes[1].set_title("Nitrogen")
axes[0].text(-0.18, 1.05, "A", transform=axes[0].transAxes, fontsize=25, fontweight="bold", va="top")

# plt.suptitle('Stage 2: Random Forest', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
c_order = ['glc', 'ac', 'lcts', 'glyc', 'succ']
n_order = ['arg', 'nh4', 'no3']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cm_carbon = confusion_matrix(le_carbon.inverse_transform(y_c_test), le_carbon.inverse_transform(xgb_c_pred), labels=c_order)
disp_carbon = ConfusionMatrixDisplay(cm_carbon, display_labels=c_order)

disp_carbon.plot(
    cmap='GnBu',
    ax=axes[0],
    colorbar=True
)

axes[0].set_title("Carbon")

cm_nitrogen = confusion_matrix(le_nitrogen.inverse_transform(y_n_test), le_nitrogen.inverse_transform(xgb_n_pred), labels=n_order)
disp_nitrogen = ConfusionMatrixDisplay(cm_nitrogen,display_labels=n_order)

disp_nitrogen.plot(cmap='GnBu',ax=axes[1],colorbar=True)

axes[1].set_title("Nitrogen")

axes[0].text(-0.18, 1.05, "B", transform=axes[0].transAxes, fontsize=25, fontweight="bold", va="top")
# plt.suptitle('Stage 2: XGBoost', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

**Full pipeline**

In [ ]:
ml.evaluate_predictions(
    y_test_source,
    rf_source_pred,
    "RF full pipeline: final source"
)

ml.evaluate_predictions(
    y_test_source,
    xgb_source_pred,
    "XGBBoost full pipeline: final source"
)

In [ ]:
env_order = ['glc', 'ac', 'lcts', 'glyc', 'succ', 'arg', 'nh4', 'no3']

cm = confusion_matrix(y_test_source, rf_source_pred, labels=env_order)
disp = ConfusionMatrixDisplay(cm, display_labels=env_order)

fig, ax = plt.subplots(figsize=(10,8))
disp.plot(cmap='GnBu', ax=ax)
# ax.set_title("Full pipeline: Random Forest", fontsize=15, pad=15)
ax.text(-0.13, 1.05, "A", transform=ax.transAxes, fontsize=25, fontweight="bold", va="top")
ax.set_xlabel("Predicted label", fontsize=13, labelpad=15)
ax.set_ylabel("True label", fontsize=13, labelpad=15)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=12)
plt.show()

In [ ]:
env_order = ['glc', 'ac', 'lcts', 'glyc', 'succ', 'arg', 'nh4', 'no3']

cm = confusion_matrix(y_test_source, xgb_source_pred, labels=env_order)
disp = ConfusionMatrixDisplay(cm, display_labels=env_order)


fig, ax = plt.subplots(figsize=(10,8))
disp.plot(cmap='GnBu', ax=ax)
# ax.set_title("Full pipeline: XGBoost", fontsize=15, pad=15)
ax.text(-0.13, 1.05, "B", transform=ax.transAxes, fontsize=25, fontweight="bold", va="top")
ax.set_xlabel("Predicted label", fontsize=13, labelpad=15)
ax.set_ylabel("True label", fontsize=13, labelpad=15)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="x", labelsize=12)
plt.show()

**Isolate Stage 2 predictors**

In [ ]:
rf_source_proba = ml.make_prediction_proba(
    X_test,
    best_rf_binary,
    best_rf_carbon,
    best_rf_nitrogen,
    le_type,
    le_source,
    le_carbon,
    le_nitrogen
)

xgb_source_proba = ml.make_prediction_proba(
    X_test,
    best_xgb_binary,
    best_xgb_carbon,
    best_xgb_nitrogen,
    le_type,
    le_source,
    le_carbon,
    le_nitrogen
)

## 4. Permutation importance

In [ ]:
feature_names = X.columns.tolist()

fi_rf_binary = ml.get_permutation_importance(
    best_rf_binary, X_test, y_test_type,
    feature_names, "Random Forest", "Stage 1", scoring="matthews_corrcoef"
)

fi_rf_carbon = ml.get_permutation_importance(
    best_rf_carbon, X_c_test, y_c_test,
    feature_names, "Random Forest", "Stage 2 carbon", scoring="f1_macro"
)

fi_rf_nitrogen = ml.get_permutation_importance(
    best_rf_nitrogen, X_n_test, y_n_test,
    feature_names, "Random Forest", "Stage 2 nitrogen", scoring="f1_macro"
)

In [ ]:
ml.plot_permutation_importance(
    fi_rf_binary,
    panel_label="A",
    style="box",
    scoring="matthews_corrcoef"
)

ml.plot_permutation_importance(
    fi_rf_carbon,
    panel_label="B",
    style="box",
    scoring="f1_macro"
)

ml.plot_permutation_importance(
    fi_rf_nitrogen,
    panel_label="C",
    style="box",
    scoring="f1_macro"
)